
<div style="text-align:center; padding: 20px 24px 10px 24px; font-family: 'Segoe UI', sans-serif;">
  
  <div style="border: 2px solid #004E9A; border-radius: 14px; padding: 22px; max-width: 860px; margin: 0 auto; box-shadow: 0 8px 22px rgba(0, 78, 154, 0.10);">
    <p style="margin: 0; font-size: 15px; letter-spacing: 0.08em; color: #4B5563;">UNIVERSIDAD TECNOLÓGICA METROPOLITANA</p>
    <h1 style="margin: 12px 0 4px 0; color: #004E9A; font-size: 32px;">Laboratorio N° 1</h1>
    <h2 style="margin: 0 0 18px 0; color: #111827; font-size: 24px; font-weight: 600;">Diagnóstico experimental de paralelismo</h2>
    <p style="margin: 6px 0;"><strong>Asignatura:</strong> INFB8090 - Computación Paralela y Distribuida</p>
    <p style="margin: 6px 0;"><strong>Carrera:</strong> Ingeniería Civil en Ciencia de Datos</p>
    <p style="margin: 6px 0;"><strong>Profesor:</strong> Dr. Ing. Michael Miranda Sandoval</p>
    <p style="margin: 6px 0;"><strong>Estudiante:</strong> Welinton Barrera Mondaca</p>
    <p style="margin: 6px 0;"><strong>Sección:</strong> 412</p>
    <p style="margin: 6px 0;"><strong>Fecha de entrega:</strong> 08 de abril de 2026</p>
    <p style="margin: 18px 0 4px 0; font-weight: 600; color: #004E9A;">Equipo utilizado para la ejecución</p>
    <p style="margin: 6px 0;">ASUS ROG Strix G513RM | AMD Ryzen 7 6800H | 16 procesadores lógicos</p>
    <p style="margin: 6px 0;">Windows 11 | Python 3.13.5 (Anaconda)</p>
  </div>
</div>



## Contexto y objetivo del laboratorio

En este laboratorio trabajé una primera aproximación experimental al paralelismo. La idea no era paralelizar de inmediato, sino observar con más cuidado cómo se comportan tareas simples en Python, medir sus tiempos de ejecución y pensar con criterio si realmente vale la pena llevarlas a un esquema paralelo o distribuido.

A partir de los tres ejercicios busqué distinguir dos cosas que me parecen centrales en esta primera etapa: primero, que una tarea lenta no se convierte automáticamente en una buena candidata para paralelizar; y segundo, que la estructura del problema importa tanto como el tiempo total. Si las unidades de trabajo son independientes, entonces recién aparece una oportunidad razonable de paralelización.

Para desarrollar el notebook usé `time.perf_counter()` como referencia de medición y mantuve una lógica reproducible, con código visible, resultados observables y comentarios interpretativos en cada sección.


In [1]:

import math
import random
import statistics
import time
from html import escape
from IPython.display import HTML, Markdown, display


def fmt_int(valor):
    return f"{valor:,}".replace(",", ".")


def fmt_float(valor, decimales=6):
    return f"{valor:.{decimales}f}"


def mostrar_tabla(filas, columnas=None):
    if not filas:
        display(Markdown("_No hay datos para mostrar._"))
        return

    if columnas is None:
        columnas = list(filas[0].keys())

    encabezado = "".join(
        f"<th style='background:#004E9A;color:white;padding:8px 10px;text-align:left;border:1px solid #D1D5DB'>{escape(str(col))}</th>"
        for col in columnas
    )

    cuerpo = []
    for fila in filas:
        celdas = "".join(
            f"<td style='padding:8px 10px;border:1px solid #D1D5DB'>{escape(str(fila.get(col, '')))}</td>"
            for col in columnas
        )
        cuerpo.append(f"<tr>{celdas}</tr>")

    html = f"""
    <table style='border-collapse:collapse; margin:12px 0; min-width: 520px;'>
      <thead><tr>{encabezado}</tr></thead>
      <tbody>{''.join(cuerpo)}</tbody>
    </table>
    """
    display(HTML(html))



## Ejercicio 1: Medición base de una tarea secuencial simple

Para partir quise construir una línea base clara con una tarea completamente secuencial. Elegí la suma acumulada de cuadrados porque es una operación simple, fácil de seguir y suficiente para observar cómo crece el tiempo cuando aumenta el tamaño de entrada.

En esta primera prueba me interesa tener una referencia base antes de pensar en cualquier mejora. Si no sé cuánto tarda una versión secuencial bien definida, entonces después tampoco tendría una base sólida para discutir si una estrategia paralela aporta algo real.


In [2]:

def suma_cuadrados_secuencial(n):
    acumulado = 0
    for i in range(n):
        acumulado += i * i
    return acumulado


tamanos_e1 = [100_000, 500_000, 1_000_000]
resultados_e1 = []

for n in tamanos_e1:
    inicio = time.perf_counter()
    resultado = suma_cuadrados_secuencial(n)
    fin = time.perf_counter()
    duracion = fin - inicio
    resultados_e1.append(
        {
            "Tamaño de entrada": fmt_int(n),
            "Resultado": str(resultado),
            "Tiempo (s)": fmt_float(duracion, 6),
            "_tiempo": duracion,
        }
    )

mostrar_tabla(
    [{k: v for k, v in fila.items() if not k.startswith("_")} for fila in resultados_e1],
    ["Tamaño de entrada", "Resultado", "Tiempo (s)"],
)


Tamaño de entrada,Resultado,Tiempo (s)
100.000,333328333350000,0.005500
500.000,41666541666750000,0.028123
1.000.000,333332833333500000,0.056454


In [3]:

tiempo_100k = resultados_e1[0]["_tiempo"]
tiempo_500k = resultados_e1[1]["_tiempo"]
tiempo_1m = resultados_e1[2]["_tiempo"]

texto_e1 = f"""
### Interpretación del Ejercicio 1

Al revisar los tiempos, veo que el costo de ejecución aumenta de forma clara a medida que crece `n`. En esta corrida, pasar de 100.000 a 500.000 elementos multiplicó el tiempo aproximadamente por **{tiempo_500k / tiempo_100k:.2f}**, y pasar de 100.000 a 1.000.000 lo multiplicó por **{tiempo_1m / tiempo_100k:.2f}**.

Para mí esta implementación funciona como una referencia secuencial base porque el cálculo depende de un único flujo de ejecución que recorre la secuencia completa de principio a fin. En esta versión no separé el trabajo en partes concurrentes ni delegué subtareas a otros procesos. Todo se resuelve en un solo recorrido, por lo que sirve como punto de comparación para cualquier análisis posterior.
"""

display(Markdown(texto_e1))



### Interpretación del Ejercicio 1

Al revisar los tiempos, veo que el costo de ejecución aumenta de forma clara a medida que crece `n`. En esta corrida, pasar de 100.000 a 500.000 elementos multiplicó el tiempo aproximadamente por **5.11**, y pasar de 100.000 a 1.000.000 lo multiplicó por **10.26**.

Para mí esta implementación funciona como una referencia secuencial base porque el cálculo depende de un único flujo de ejecución que recorre la secuencia completa de principio a fin. En esta versión no separé el trabajo en partes concurrentes ni delegué subtareas a otros procesos. Todo se resuelve en un solo recorrido, por lo que sirve como punto de comparación para cualquier análisis posterior.



## Ejercicio 2: Comparación de variantes de procesamiento y discusión sobre paralelización

En este segundo ejercicio comparé dos formas de resolver la misma tarea sobre una colección de datos. La primera variante recorre toda la lista elemento a elemento. La segunda hace el mismo trabajo, pero separando la lista en bloques para dejar más visible la existencia de subconjuntos que podrían tratarse por separado.

Lo importante acá no era lograr más velocidad solo por cambiar la forma del código. De hecho, si ambas versiones siguen ejecutándose secuencialmente, puede que la diferencia sea mínima o incluso que la variante por bloques tarde un poco más. Lo interesante es mirar la estructura del problema y preguntarse si esos bloques son suficientemente independientes como para pensar después en paralelización.


In [4]:

datos = list(range(1, 400_001))


def carga_elemental(x):
    return math.sqrt(x) + math.log(x + 1)


def procesar_secuencial(datos):
    total = 0.0
    for x in datos:
        total += carga_elemental(x)
    return total


def procesar_por_bloques(datos, tam_bloque=50_000):
    total = 0.0
    for inicio in range(0, len(datos), tam_bloque):
        bloque = datos[inicio:inicio + tam_bloque]
        subtotal = 0.0
        for x in bloque:
            subtotal += carga_elemental(x)
        total += subtotal
    return total


inicio = time.perf_counter()
total_secuencial = procesar_secuencial(datos)
tiempo_secuencial = time.perf_counter() - inicio

inicio = time.perf_counter()
total_bloques = procesar_por_bloques(datos)
tiempo_bloques = time.perf_counter() - inicio

diferencia_totales = abs(total_secuencial - total_bloques)

resultados_e2 = [
    {
        "Variante": "Procesamiento secuencial",
        "Total obtenido": fmt_float(total_secuencial, 4),
        "Tiempo (s)": fmt_float(tiempo_secuencial, 6),
    },
    {
        "Variante": "Procesamiento por bloques",
        "Total obtenido": fmt_float(total_bloques, 4),
        "Tiempo (s)": fmt_float(tiempo_bloques, 6),
    },
    {
        "Variante": "Diferencia absoluta entre totales",
        "Total obtenido": fmt_float(diferencia_totales, 10),
        "Tiempo (s)": "-",
    },
]

mostrar_tabla(resultados_e2, ["Variante", "Total obtenido", "Tiempo (s)"])


Variante,Total obtenido,Tiempo (s)
Procesamiento secuencial,173414832.7605,0.064986
Procesamiento por bloques,173414832.7605,0.061771
Diferencia absoluta entre totales,0.0000024438,-


In [5]:

if tiempo_bloques < tiempo_secuencial:
    comparacion_e2 = (
        f"En esta ejecución, la versión por bloques fue levemente más rápida, con una diferencia de "
        f"**{abs(tiempo_secuencial - tiempo_bloques):.6f} s**."
    )
else:
    comparacion_e2 = (
        f"En esta ejecución, la versión por bloques tardó un poco más, con una diferencia de "
        f"**{abs(tiempo_secuencial - tiempo_bloques):.6f} s** respecto de la versión secuencial."
    )

texto_e2 = f"""
### Interpretación del Ejercicio 2

{comparacion_e2}

Más allá del tiempo exacto, lo que me parece relevante es que ambas versiones entregan el mismo resultado, pero la segunda deja explícita una idea importante: la lista puede dividirse en bloques relativamente independientes. Cada bloque se procesa con la misma lógica y luego sus subtotales se suman al final.

Desde mi punto de vista, ahí sí aparece una oportunidad razonable de paralelización. Si cada bloque se ejecutara por separado en procesos o tareas distintas, habría poco acoplamiento durante el cálculo y la coordinación principal quedaría reducida a combinar los subtotales al final. Eso sí, tendría sentido paralelizar solamente si el costo computacional por bloque justifica la sobrecarga adicional de crear y coordinar procesos.
"""

display(Markdown(texto_e2))



### Interpretación del Ejercicio 2

En esta ejecución, la versión por bloques fue levemente más rápida, con una diferencia de **0.003215 s**.

Más allá del tiempo exacto, lo que me parece relevante es que ambas versiones entregan el mismo resultado, pero la segunda deja explícita una idea importante: la lista puede dividirse en bloques relativamente independientes. Cada bloque se procesa con la misma lógica y luego sus subtotales se suman al final.

Desde mi punto de vista, ahí sí aparece una oportunidad razonable de paralelización. Si cada bloque se ejecutara por separado en procesos o tareas distintas, habría poco acoplamiento durante el cálculo y la coordinación principal quedaría reducida a combinar los subtotales al final. Eso sí, tendría sentido paralelizar solamente si el costo computacional por bloque justifica la sobrecarga adicional de crear y coordinar procesos.



## Ejercicio 3: Caso aplicado a ciencia de datos y decisión diagnóstica

En este ejercicio trabajé con un escenario que ya se parece más a una situación real de ciencia de datos: varios lotes de información que pueden tratarse como unidades de trabajo separadas. Este tipo de estructura es común cuando se procesan archivos independientes, particiones de datos, validaciones por bloques o evaluaciones repetidas sobre subconjuntos.

La meta acá no fue solo medir el tiempo total, sino también decidir qué estrategia computacional parece más razonable. Para responder eso me fijé en cuatro criterios: independencia de las tareas, volumen de datos, costo del cálculo y necesidad de coordinación entre unidades de trabajo.


In [6]:

random.seed(42)
lotes = []

for _ in range(8):
    lote = [random.uniform(0, 100) for _ in range(120_000)]
    lotes.append(lote)


def resumir_lote(lote):
    media = statistics.fmean(lote)
    maximo = max(lote)
    minimo = min(lote)
    return {
        "media": media,
        "maximo": maximo,
        "minimo": minimo,
    }


inicio = time.perf_counter()
resumenes = []

for indice, lote in enumerate(lotes, start=1):
    resumen = resumir_lote(lote)
    resumenes.append(
        {
            "Lote": indice,
            "Media": fmt_float(resumen["media"], 3),
            "Mínimo": fmt_float(resumen["minimo"], 3),
            "Máximo": fmt_float(resumen["maximo"], 3),
        }
    )

fin = time.perf_counter()
tiempo_total_e3 = fin - inicio

mostrar_tabla(resumenes, ["Lote", "Media", "Mínimo", "Máximo"])
display(Markdown(f"**Tiempo total de procesamiento:** {tiempo_total_e3:.6f} s"))


Lote,Media,Mínimo,Máximo
1,49.973,0.001,100.000
2,50.090,0.002,99.999
3,49.996,0.000,100.000
4,50.145,0.001,99.999
5,49.948,0.000,99.999
6,49.849,0.000,99.998
7,50.073,0.001,100.000
8,49.882,0.001,100.000


**Tiempo total de procesamiento:** 0.036482 s

In [7]:

texto_e3 = f"""
### Decisión diagnóstica del Ejercicio 3

En este caso, mi decisión es que el problema se presta **más a paralelismo en un mismo equipo** que a una estrategia distribuida entre varios equipos.

La razón principal es que los lotes son independientes entre sí: cada uno puede resumirse sin depender del resultado de los demás, lo que hace bastante natural repartir el trabajo. Sin embargo, el volumen del ejemplo sigue siendo manejable y la coordinación necesaria es baja, por lo que no veo una justificación fuerte para distribuir el procesamiento entre varias máquinas. En un escenario así, una solución distribuida agregaría complejidad de comunicación, partición y control que probablemente no se recupera con el tamaño del problema planteado acá.

Si este mismo patrón creciera mucho más en volumen, o si los lotes provinieran de múltiples fuentes y no cupieran bien en un solo equipo, entonces sí tendría sentido pensar en computación distribuida. Pero con la evidencia de este ejercicio, me parece más razonable hablar de paralelismo local como primera opción.
"""

display(Markdown(texto_e3))



### Decisión diagnóstica del Ejercicio 3

En este caso, mi decisión es que el problema se presta **más a paralelismo en un mismo equipo** que a una estrategia distribuida entre varios equipos.

La razón principal es que los lotes son independientes entre sí: cada uno puede resumirse sin depender del resultado de los demás, lo que hace bastante natural repartir el trabajo. Sin embargo, el volumen del ejemplo sigue siendo manejable y la coordinación necesaria es baja, por lo que no veo una justificación fuerte para distribuir el procesamiento entre varias máquinas. En un escenario así, una solución distribuida agregaría complejidad de comunicación, partición y control que probablemente no se recupera con el tamaño del problema planteado acá.

Si este mismo patrón creciera mucho más en volumen, o si los lotes provinieran de múltiples fuentes y no cupieran bien en un solo equipo, entonces sí tendría sentido pensar en computación distribuida. Pero con la evidencia de este ejercicio, me parece más razonable hablar de paralelismo local como primera opción.



## Conclusiones

Después de ejecutar los tres ejercicios, me quedó más claro que el paralelismo no se decide por intuición ni por costumbre, sino a partir de cómo está estructurado el problema. En el primer ejercicio tuve una tarea claramente secuencial, útil como línea base. En el segundo, aunque no paralelicé, ya apareció una organización por bloques que deja ver independencia entre partes. En el tercero, esa independencia fue todavía más evidente al trabajar con lotes separados.

También me pareció importante comprobar que medir antes de optimizar no es un detalle menor. Si uno no registra tiempos ni observa cómo se comporta el cálculo, corre el riesgo de proponer soluciones más complejas sin una necesidad real. En otras palabras, primero hay que entender el problema y después decidir si vale la pena paralelizar, distribuir o mantener una solución secuencial.



## Reflexión diagnóstica final

### ¿Qué características debe tener un problema para que el paralelismo resulte pertinente?

Desde mi punto de vista, un problema se vuelve una buena candidata para paralelismo cuando puede dividirse en unidades de trabajo relativamente independientes, cuando el costo computacional de cada parte es suficiente para justificar la coordinación adicional y cuando la combinación de resultados finales no introduce una dependencia demasiado fuerte. Si todas las partes del cálculo dependen unas de otras a cada momento, entonces paralelizar deja de ser tan conveniente.

### ¿Por qué no basta con que un programa demore para concluir que debe paralelizarse?

No basta con que un programa sea lento porque la lentitud por sí sola no dice nada sobre la estructura interna del problema. Puede pasar que el cuello de botella provenga de una parte que no se puede separar bien, o que la sobrecarga de coordinar procesos termine siendo mayor que la ganancia esperada. Por eso medir es necesario, pero también hay que analizar cómo está organizado el trabajo y si realmente existe independencia entre subtareas.

### ¿En qué tipos de tareas de ciencia de datos imagino que el paralelismo o la distribución podrían aportar mayor valor?

Creo que puede aportar bastante valor en tareas como procesamiento por particiones de datos, evaluación de modelos sobre distintos conjuntos, extracción de características desde archivos independientes, simulaciones repetidas, validación cruzada y preprocesamiento masivo. En todos esos casos suele existir trabajo repetitivo que puede repartirse sin una dependencia fuerte entre partes. La distribución, en cambio, la imagino más útil cuando el volumen de datos o la carga total supera con claridad la capacidad de un solo equipo.

### ¿Qué aprendí en esta primera sesión sobre la importancia de medir antes de optimizar?

En esta primera sesión confirmé que optimizar sin evidencia es una mala práctica. Medir me permitió tener una base concreta para comparar, observar patrones de crecimiento y sostener mejor cualquier decisión técnica. Más que pensar el paralelismo como una solución universal, entendí que conviene verlo como una decisión fundamentada en datos, estructura del problema y costo real de implementación.
